<a href="https://colab.research.google.com/github/Odewenu/network-anomaly-detection/blob/main/Anomaly_Detection_in_Network_Traffic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Anomaly_Detection_in_Network_Traffic

####Data Cleaning and Preprocessing

In [1]:
# Install and import the libraries we need
!pip install -q kagglehub pyarrow

import os
import numpy as np
import pandas as pd
import joblib
import kagglehub
from sklearn.preprocessing import StandardScaler

# Download UNSW-NB15 from Kaggle and load the train and test files
folder = kagglehub.dataset_download("dhoogla/unswnb15")
train = pd.read_parquet(os.path.join(folder, "UNSW_NB15_training-set.parquet"))
test = pd.read_parquet(os.path.join(folder, "UNSW_NB15_testing-set.parquet"))

print("Original train shape:", train.shape)
print("Original test shape:", test.shape)

# Check for missing values (expected: 0)
print("\nMissing values:")
print("Train:", train.isnull().sum().sum())
print("Test:", test.isnull().sum().sum())

# Create the output folders and save the raw column schemas
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

pd.DataFrame({"column": train.columns, "type": train.dtypes.astype(str).values}).to_csv("data/raw/train_schema.csv", index=False)
pd.DataFrame({"column": test.columns, "type": test.dtypes.astype(str).values}).to_csv("data/raw/test_schema.csv", index=False)

# Convert category-type columns to plain text so we can edit them later
cat_cols = ["proto", "service", "state"]
for df in (train, test):
    for c in cat_cols + ["attack_cat"]:
        df[c] = df[c].astype(str)

# Class balance BEFORE cleaning (attacks are ~68%, which is unrealistic)
print("\nClass balance BEFORE duplicate removal:")
before_counts = train["label"].value_counts()
print(before_counts)
print("\nClass percentages:")
print((before_counts / len(train) * 100).round(2))

100%|██████████| 11.7M/11.7M [00:00<00:00, 120MB/s]

Extracting files...


Original train shape: (175341, 36)
Original test shape: (82332, 36)

Missing values:
Train: 0
Test: 0

Class balance BEFORE duplicate removal:
label
1    119341
0     56000
Name: count, dtype: int64

Class percentages:
label
1    68.06
0    31.94
Name: count, dtype: float64


In [2]:
print("\nDuplicates:")
print("Train:", train.duplicated().sum())
print("Test:", test.duplicated().sum())

# Remove duplicate rows inside each set (stops the model memorising repeats)
train = train.drop_duplicates().reset_index(drop=True)
test = test.drop_duplicates().reset_index(drop=True)

print("\nAfter removing duplicates:")
print("Train:", train.shape)
print("Test:", test.shape)

# Turn every row into a text "key" so train and test rows can be compared
common_columns = [c for c in train.columns if c in test.columns]
train_keys = train[common_columns].astype(str).agg("|".join, axis=1)
test_keys = test[common_columns].astype(str).agg("|".join, axis=1)
train_key_set = set(train_keys)

# Drop test rows that also appear in train (they would be "already seen" data)
test_keep = ~test_keys.isin(train_key_set)
removed_overlap = (~test_keep).sum()
test = test.loc[test_keep].reset_index(drop=True)

print("\nRemoved train/test overlapping rows:", removed_overlap)
print("Test shape after removing overlap:", test.shape)


Duplicates:
Train: 78519
Test: 32361

After removing duplicates:
Train: (96822, 36)
Test: (49971, 36)

Removed train/test overlapping rows: 1571
Test shape after removing overlap: (48400, 36)


In [3]:
# Create new features from existing columns (+1 avoids dividing by zero)
def add_features(df):
    df = df.copy()
    if "sbytes" in df.columns and "dbytes" in df.columns:
        df["total_bytes"] = df["sbytes"] + df["dbytes"]          # all bytes sent both ways
        df["byte_ratio"] = df["sbytes"] / (df["dbytes"] + 1)     # source vs destination bytes
    if "spkts" in df.columns and "dpkts" in df.columns:
        df["total_packets"] = df["spkts"] + df["dpkts"]          # all packets both ways
        df["packet_ratio"] = df["spkts"] / (df["dpkts"] + 1)     # source vs destination packets
    if "sbytes" in df.columns and "spkts" in df.columns:
        df["source_bytes_per_packet"] = df["sbytes"] / (df["spkts"] + 1)
    if "dbytes" in df.columns and "dpkts" in df.columns:
        df["destination_bytes_per_packet"] = df["dbytes"] / (df["dpkts"] + 1)
    return df

train = add_features(train)
test = add_features(test)

print("\nFeature engineering completed.")
print("New train shape:", train.shape)
print("New test shape:", test.shape)

# Outlier analysis using the IQR rule. We only MEASURE outliers here and do
# not delete them, because in intrusion detection extreme values are often attacks.
numeric_cols = train.select_dtypes(include="number").columns
numeric_cols = [c for c in numeric_cols if c != "label"]

outlier_summary = []
for c in numeric_cols:
    q1 = train[c].quantile(0.25)
    q3 = train[c].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((train[c] < lower) | (train[c] > upper)).sum()
    outlier_summary.append({"column": c, "outlier_count": int(count), "outlier_percentage": round(count / len(train) * 100, 2)})

# Save the full outlier report and show the 10 worst columns
outlier_summary = pd.DataFrame(outlier_summary)
outlier_summary.to_csv("data/processed/outlier_summary.csv", index=False)

print("\nOutlier analysis completed.")
print(outlier_summary.sort_values("outlier_count", ascending=False).head(10))


Feature engineering completed.
New train shape: (96822, 42)
New test shape: (48400, 42)

Outlier analysis completed.
                          column  outlier_count  outlier_percentage
25              ct_src_dport_ltm          21633               22.34
17                          dwin          21259               21.96
14                          swin          21028               21.72
4                         dbytes          20076               20.73
36  destination_bytes_per_packet          19531               20.17
22                         dmean          19247               19.88
32                    byte_ratio          19149               19.78
29              ct_flw_http_mthd          17203               17.77
23                   trans_depth          17203               17.77
31                   total_bytes          17156               17.72


In [4]:
# Find columns with heavy right skew (skewness above 5) that can be log-transformed
num_cols = train.select_dtypes(include="number").columns.drop("label")
skew = train[num_cols].skew()
skewed_cols = [c for c in skew[skew > 5].index if train[c].min() >= 0 and test[c].min() >= 0 and train[c].nunique() > 2]

# log(1 + x) pulls huge values closer to the rest of the data
for df in (train, test):
    df[skewed_cols] = np.log1p(df[skewed_cols])

print("\nLog-transformed columns:", len(skewed_cols))
print(skewed_cols)

# Check that the skew really went down
after_skew = train[skewed_cols].skew()
if len(after_skew) > 0:
    print("\nHighest skewness after transformation:", round(after_skew.abs().max(), 2))

# proto has 133 values, so keep the 5 most common (from TRAIN only) and group the rest
top_proto = train["proto"].value_counts().head(5).index
for df in (train, test):
    df["proto"] = df["proto"].where(df["proto"].isin(top_proto), "other")

# Turn text columns into 0/1 columns the model can read (one-hot encoding)
train = pd.get_dummies(train, columns=cat_cols, dtype=int)
test = pd.get_dummies(test, columns=cat_cols, dtype=int)

# Make sure test has exactly the same columns as train, in the same order
test = test.reindex(columns=train.columns, fill_value=0)

print("\nAfter encoding:")
print("Train:", train.shape)
print("Test:", test.shape)


Log-transformed columns: 25
['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'synack', 'trans_depth', 'response_body_len', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'total_bytes', 'byte_ratio', 'total_packets', 'packet_ratio']

Highest skewness after transformation: 4.75

After encoding:
Train: (96822, 67)
Test: (48400, 67)


/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)


In [5]:
# Pick the real number columns to scale (not the label, attack_cat, or 0/1 columns)
to_scale = [c for c in train.columns if c not in ("label", "attack_cat") and train[c].nunique() > 2]

# Fit the scaler on TRAIN only, then apply it to both (prevents data leakage)
scaler = StandardScaler()
train[to_scale] = scaler.fit_transform(train[to_scale])
test[to_scale] = scaler.transform(test[to_scale])

print("\nScaled columns:", len(to_scale))

# Class balance AFTER cleaning (should be close to 50/50)
print("\nClass balance AFTER cleaning:")
after_counts = train["label"].value_counts()
print(after_counts)
print("\nClass percentages:")
print((after_counts / len(train) * 100).round(2))

# ---- NEW: save attack_cat separately, then remove it from the model files ----
# attack_cat gives away the answer, so it must not be used as an input.
# Row order matches the clean files, so it can still be used later for analysis.
train[["attack_cat"]].to_parquet("data/processed/train_attack_cat.parquet", index=False)
test[["attack_cat"]].to_parquet("data/processed/test_attack_cat.parquet", index=False)
train = train.drop(columns=["attack_cat"])
test = test.drop(columns=["attack_cat"])
print("\nFinal shapes (attack_cat removed):", train.shape, test.shape)
# ---- END NEW ----

# Shrink float64 to float32 so the files are small enough for GitHub
def small(df):
    df = df.copy()
    float_cols = df.select_dtypes(include="float64").columns
    df[float_cols] = df[float_cols].astype("float32")
    return df

# Save the cleaned data, the fitted scaler (needed for deployment) and the schema
small(train).to_parquet("data/processed/train_clean.parquet", index=False, compression="gzip")
small(test).to_parquet("data/processed/test_clean.parquet", index=False, compression="gzip")
joblib.dump(scaler, "data/processed/scaler.pkl")
pd.DataFrame({"column": train.columns, "type": train.dtypes.astype(str).values}).to_csv("data/processed/data_schema.csv", index=False)

# List everything that was saved
print("\nFinal files:")
for root, dirs, files in os.walk("data"):
    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path) / 1e6
        print(path, "-", round(size, 2), "MB")

print("\n====================================")
print("PHASE 1 DATA CLEANING COMPLETED")
print("====================================")


Scaled columns: 36

Class balance AFTER cleaning:
label
0    48894
1    47928
Name: count, dtype: int64

Class percentages:
label
0    50.5
1    49.5
Name: count, dtype: float64

Final shapes (attack_cat removed): (96822, 66) (48400, 66)

Final files:
data/raw/test_schema.csv - 0.0 MB
data/raw/train_schema.csv - 0.0 MB
data/processed/test_attack_cat.parquet - 0.01 MB
data/processed/train_clean.parquet - 7.15 MB
data/processed/data_schema.csv - 0.0 MB
data/processed/outlier_summary.csv - 0.0 MB
data/processed/scaler.pkl - 0.0 MB
data/processed/train_attack_cat.parquet - 0.03 MB
data/processed/test_clean.parquet - 3.69 MB

PHASE 1 DATA CLEANING COMPLETED


####Model Training

####Model Deployment